### Embedding Comparison


In [2]:
from pathlib import Path
from gensim.models import Word2Vec, FastText

from utils import load_sets, clean, tokenise

In [3]:
train_set, val_set, test_set = load_sets()

print("Train:", len(train_set))
print("Val:", len(val_set))
print("Test:", len(test_set))

train_set.head()

Train: 199222
Val: 22136
Test: 24596


,input,response
0,"hello, my little friends. where can i find a c...",nfi #windows
1,i heard fedora was better than ubuntu,than join #fedora you hear things join #ubunt...
2,"if your doing hardware hacking yes, if your a ...",thanks. :)
3,hi everyone! that joke isn't funny anymore.,why? cuz he died?
4,so where is the boot loader like grub stored i...,<PATH> is inside of / if there is no separate...


In [4]:
def build_sentences(df):
    sentences = []

    for _, row in df.iterrows():
        input_text = clean(str(row["input"]))
        response_text = clean(str(row["response"]))

        input_tokens = tokenise(input_text)
        response_tokens = tokenise(response_text)

        if input_tokens:
            sentences.append(input_tokens)
        if response_tokens:
            sentences.append(response_tokens)

    return sentences

sentences = build_sentences(train_set)

print("Number of tokenized sentences:", len(sentences))
print("Example:", sentences[0][:20] if sentences else "No data")

Number of tokenized sentences: 398444
Example: ['hello,', 'my', 'little', 'friends.', 'where', 'can', 'i', 'find', 'a', 'channel', 'for', 'internetexplorer.application', '?', '(activex)']


In [5]:


Path("embeddings").mkdir(exist_ok=True)

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1
)

w2v_model.save("embeddings/word2vec.model")

print("Word2Vec trained and saved as word2vec.model")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Word2Vec trained and saved as word2vec.model


In [6]:


Path("embeddings").mkdir(exist_ok=True)

fasttext_model = FastText(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1
)

fasttext_model.save("embeddings/fasttext.model")

print("FastText trained and saved as fasttext.model")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


FastText trained and saved as fasttext.model


### Replicating Model A loading

In [7]:
from utils import load_sets, load_vocab, load_config, get_dataloaders

train_set, val_set, test_set = load_sets()
vocab = load_vocab()
config = load_config()

train_loader, val_loader, test_loader = get_dataloaders(
    train_set, val_set, test_set,
    vocab,
    config['MAX_LENGTH'],
    config['BATCH_SIZE']
)

### Pretrained from same vocab

In [9]:
import numpy as np
import torch
from gensim.models import Word2Vec, FastText

EMBEDDING_TYPE = "scratch"   # "scratch", "word2vec", "fasttext"

w2v_model = Word2Vec.load("embeddings/word2vec.model")
ft_model = FastText.load("embeddings/fasttext.model")

def build_embedding_matrix(vocab, model, embed_dim, pad_idx):
    matrix = np.random.normal(scale=0.6, size=(len(vocab), embed_dim)).astype(np.float32)
    matrix[pad_idx] = np.zeros(embed_dim, dtype=np.float32)

    for token, idx in vocab.items():
        if token in model.wv:
            matrix[idx] = model.wv[token]

    return torch.tensor(matrix, dtype=torch.float32)

ENCODER_EMBED_DIM = 100
DECODER_EMBED_DIM = 100

w2v_matrix_enc = build_embedding_matrix(vocab, w2v_model, ENCODER_EMBED_DIM, config["PAD_IDX"])
ft_matrix_enc  = build_embedding_matrix(vocab, ft_model,  ENCODER_EMBED_DIM, config["PAD_IDX"])

w2v_matrix_dec = build_embedding_matrix(vocab, w2v_model, DECODER_EMBED_DIM, config["PAD_IDX"])
ft_matrix_dec  = build_embedding_matrix(vocab, ft_model,  DECODER_EMBED_DIM, config["PAD_IDX"])

### Encoder/Decoder/Seq 2 Seq

In [10]:
import torch
import torch.nn as nn
import random

class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        padding_idx,
        hidden_dim,
        num_layers,
        dropout_proba,
        embedding_type="scratch",
        pretrained_matrix=None
    ):
        super().__init__()

        if embedding_type == "scratch":
            self.embedding = nn.Embedding(
                vocab_size,
                embed_dim,
                padding_idx=padding_idx
            )
        else:
            self.embedding = nn.Embedding.from_pretrained(
                pretrained_matrix,
                freeze=False,
                padding_idx=padding_idx
            )

        self.dropout = nn.Dropout(dropout_proba)
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_proba if num_layers > 1 else 0
        )

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, hidden = self.gru(embedded)
        return hidden


class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        padding_idx,
        hidden_dim,
        num_layers,
        dropout_proba,
        embedding_type="scratch",
        pretrained_matrix=None
    ):
        super().__init__()

        if embedding_type == "scratch":
            self.embedding = nn.Embedding(
                vocab_size,
                embed_dim,
                padding_idx=padding_idx
            )
        else:
            self.embedding = nn.Embedding.from_pretrained(
                pretrained_matrix,
                freeze=False,
                padding_idx=padding_idx
            )

        self.dropout = nn.Dropout(dropout_proba)
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_proba if num_layers > 1 else 0
        )
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        x = x.unsqueeze(1)
        embedded = self.dropout(self.embedding(x))
        output, hidden = self.gru(embedded, hidden)
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_proba=0.5):
        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        vocab_size = self.decoder.fc_out.out_features

        outputs = torch.zeros(batch_size, tgt_len, vocab_size).to(self.device)

        hidden = self.encoder(src)
        decoder_input = tgt[:, 0]

        for t in range(1, tgt_len):
            output, hidden = self.decoder(decoder_input, hidden)
            outputs[:, t, :] = output

            top1 = output.argmax(1)
            teacher_force = random.random() < teacher_forcing_proba
            decoder_input = tgt[:, t] if teacher_force else top1

        return outputs

### Train Eval

In [11]:
from tqdm import tqdm
import torch
import torch.nn as nn

def train_epoch(model, loader, optimizer, criterion, device, clip=1.0, teacher_forcing_proba=0.5):
    model.train()
    total_loss = 0

    progress = tqdm(loader, desc="Training", leave=False)

    for src, tgt_in, tgt_out in progress:
        src = src.to(device)
        tgt_in = tgt_in.to(device)
        tgt_out = tgt_out.to(device)

        optimizer.zero_grad()

        output = model(src, tgt_in, teacher_forcing_proba=teacher_forcing_proba)
        output = output[:, 1:, :].reshape(-1, output.shape[-1])
        tgt_out = tgt_out[:, 1:].reshape(-1)

        loss = criterion(output, tgt_out)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        total_loss += loss.item()
        progress.set_postfix(loss=loss.item())

    return total_loss / len(loader)


def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    progress = tqdm(loader, desc="Validation", leave=False)

    with torch.no_grad():
        for src, tgt_in, tgt_out in progress:
            src = src.to(device)
            tgt_in = tgt_in.to(device)
            tgt_out = tgt_out.to(device)

            output = model(src, tgt_in, teacher_forcing_proba=0.0)
            output = output[:, 1:, :].reshape(-1, output.shape[-1])
            tgt_out = tgt_out[:, 1:].reshape(-1)

            loss = criterion(output, tgt_out)
            total_loss += loss.item()
            progress.set_postfix(loss=loss.item())

    return total_loss / len(loader)

### Model Building

In [12]:
def make_model(embedding_type, vocab, config, w2v_model, ft_model, device):
    ENCODER_EMBED_DIM = 100
    DECODER_EMBED_DIM = 100

    ENCODER_HIDDEN_DIM = 256
    ENCODER_NUM_LAYERS = 2
    ENCODER_DROPOUT_PROBA = 0.3

    DECODER_HIDDEN_DIM = 256
    DECODER_NUM_LAYERS = 2
    DECODER_DROPOUT_PROBA = 0.3

    if embedding_type == "word2vec":
        encoder_matrix = build_embedding_matrix(vocab, w2v_model, ENCODER_EMBED_DIM, config["PAD_IDX"])
        decoder_matrix = build_embedding_matrix(vocab, w2v_model, DECODER_EMBED_DIM, config["PAD_IDX"])
    elif embedding_type == "fasttext":
        encoder_matrix = build_embedding_matrix(vocab, ft_model, ENCODER_EMBED_DIM, config["PAD_IDX"])
        decoder_matrix = build_embedding_matrix(vocab, ft_model, DECODER_EMBED_DIM, config["PAD_IDX"])
    else:
        encoder_matrix = None
        decoder_matrix = None

    model = Seq2Seq(
        encoder=Encoder(
            vocab_size=config["VOCAB_SIZE"],
            embed_dim=ENCODER_EMBED_DIM,
            padding_idx=config["PAD_IDX"],
            hidden_dim=ENCODER_HIDDEN_DIM,
            num_layers=ENCODER_NUM_LAYERS,
            dropout_proba=ENCODER_DROPOUT_PROBA,
            embedding_type=embedding_type,
            pretrained_matrix=encoder_matrix
        ),
        decoder=Decoder(
            vocab_size=config["VOCAB_SIZE"],
            embed_dim=DECODER_EMBED_DIM,
            padding_idx=config["PAD_IDX"],
            hidden_dim=DECODER_HIDDEN_DIM,
            num_layers=DECODER_NUM_LAYERS,
            dropout_proba=DECODER_DROPOUT_PROBA,
            embedding_type=embedding_type,
            pretrained_matrix=decoder_matrix
        ),
        device=device
    ).to(device)

    return model

### Experiment Runner

In [13]:
def run_experiment(embedding_type, train_loader, val_loader, vocab, config, w2v_model, ft_model, device, epochs=10):
    print(f"\nStarting experiment: {embedding_type}\n")

    model = make_model(embedding_type, vocab, config, w2v_model, ft_model, device)

    criterion = nn.CrossEntropyLoss(ignore_index=config["PAD_IDX"])
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss = evaluate_epoch(model, val_loader, criterion, device)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f"{embedding_type} | Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    return {
        "embedding": embedding_type,
        "final_train_loss": train_losses[-1],
        "final_val_loss": val_losses[-1],
        "train_losses": train_losses,
        "val_losses": val_losses,
        "model": model
    }

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

all_results = []

for emb_type in ["scratch", "word2vec", "fasttext"]:
    result = run_experiment(
        embedding_type=emb_type,
        train_loader=train_loader,
        val_loader=val_loader,
        vocab=vocab,
        config=config,
        w2v_model=w2v_model,
        ft_model=ft_model,
        device=device,
        epochs=10
    )
    all_results.append(result)


Starting experiment: scratch



scratch | Epoch 1/10 | Train Loss: 6.2649 | Val Loss: 6.1834


scratch | Epoch 2/10 | Train Loss: 6.0185 | Val Loss: 6.1151


scratch | Epoch 3/10 | Train Loss: 5.8931 | Val Loss: 6.0937


scratch | Epoch 4/10 | Train Loss: 5.7959 | Val Loss: 6.0857


...

fasttext | Epoch 6/10 | Train Loss: 5.4654 | Val Loss: 6.0217


fasttext | Epoch 7/10 | Train Loss: 5.3972 | Val Loss: 6.0417


fasttext | Epoch 8/10 | Train Loss: 5.3478 | Val Loss: 6.0678


fasttext | Epoch 9/10 | Train Loss: 5.2924 | Val Loss: 6.0945


fasttext | Epoch 10/10 | Train Loss: 5.2535 | Val Loss: 6.1174


### Comparison

In [15]:
import pandas as pd

comparison_df = pd.DataFrame([
    {
        "Embedding": r["embedding"],
        "Train Loss": r["final_train_loss"],
        "Val Loss": r["final_val_loss"]
    }
    for r in all_results
])

comparison_df

,Embedding,Train Loss,Val Loss
0,scratch,5.455112,6.152511
1,word2vec,5.224731,6.147417
2,fasttext,5.253455,6.117411
